In [50]:

from pathlib import Path
import datetime
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio as rio


# reload the module
import importlib

import class_downloadGEE as dlGEE
importlib.reload(dlGEE)


from tqdm import tqdm

In [51]:
ee.Initialize(project='agbd-gedi')
ee.Authenticate()

True

# Getting latitude, longitude and date information from GEDI data

In [52]:
wkdir = Path(r"G:\My Drive\GEE_GEDI_csvs")
csv_list = list(wkdir.glob("*.csv"))


In [53]:
df = pd.read_csv(csv_list[0])

In [54]:
df['date'] = df['date'].astype(str).str[:6]  # Ensure only the first 6 characters
df['date'] = pd.to_datetime(df['date'], format='%Y%m', errors='coerce')

In [55]:
len(df)

3684

# Downloading MSI/Sentinel-2 data

In [56]:
results = []


for i in tqdm(range(len(df))):

    dataset = ee.ImageCollection('COPERNICUS/S2_SR') \
        .filterBounds(ee.Geometry.Point(df.iloc[i].x , df.iloc[i].y)) \
        .filterDate(str(df.iloc[i].date)[:10], str(df.iloc[i].date + pd.DateOffset(months=1))[:10]) \
        .sort('CLOUDY_PIXEL_PERCENTAGE') \
        .mosaic() \
    
            # Extract information from the dataset
    info = dataset.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=(ee.Geometry.Point(df.iloc[i].x , df.iloc[i].y)),
        scale=10
    ).getInfo()
    results.append({'x': df.iloc[i].x, 'y': df.iloc[i].y, 'date': df.iloc[i].date, 'agbd': df.iloc[i].agbd, **info})

    
df_results = pd.DataFrame(results)




 16%|█▌        | 581/3684 [14:45<2:10:40,  2.53s/it]